In [1]:
import subprocess
import os
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import time

In [2]:
base_dir = os.getcwd()
esm2_dir = "results_esm2"
results_fasta_dir = os.path.join(esm2_dir, "fasta_files")
genetic_functions_dir = os.path.join(base_dir, "genetic")
initial_pop_dir = os.path.join(base_dir, "initial_population")

## **Functions**

Function to create a FASTA file from a sequence

In [3]:
def create_fasta(sequence, fasta_results_dir):
    # 1. Use the sequence as the filename
    filename = f"{sequence}.fasta"
    fasta_path = os.path.join(fasta_results_dir, filename)
    
    # 2. Write file using standard FASTA format
    with open(fasta_path, 'w') as f:
        f.write(f">{sequence}\n")  # Header line starts with '>'
        f.write(f"{sequence}\n")   # Sequence in the next line
        
    print(f"FASTA file successfully saved at: {fasta_path}")
    
    # Return the path for downstream usage
    return fasta_path

Function to predict maps contact with esm2 

In [4]:
def generate_contact_map(fasta_path, sequence):
    """
    Executes an external ESM-2 script to generate a contact map from a FASTA file.
    The resulting CSV is stored in results_esm2/contact_maps/. If the file already
    exists, the computation is skipped.
    """
    esm2_dir = "results_esm2"
    contact_maps_dir = os.path.join(esm2_dir, "contact_maps")
    os.makedirs(contact_maps_dir, exist_ok=True)
    
    csv_path = os.path.join(contact_maps_dir, f"{sequence}.csv")
    
    if os.path.exists(csv_path):
        print(f"Skipping: contact map already exists at {csv_path}")
        return csv_path
    
    python_env = "/home/biocomp/anaconda3/envs/esmfold/bin/python"
    script_path = os.path.join("esm2", "get_contact_map.py")
    
    print(f"Running ESM-2 contact map generation for sequence: {sequence}")
    
    try:
        subprocess.run([python_env, script_path, fasta_path, csv_path], check=True)
        print(f"Contact map successfully generated at: {csv_path}")
        return csv_path
        
    except subprocess.CalledProcessError:
        print("Error: ESM-2 script execution failed.")
    except Exception as e:
        print(f"Unexpected error: {e}")
    
    return None

Auxiliary functions for fitness

In [5]:
def mask_obvious_contacts(csv_path, min_separation=6):
    """
    Loads a contact map CSV, ignores the obvious short-range contacts,
    and returns a filtered numpy matrix.
    """
    print(f" Loading matrix and filtering obvious contacts (separation < {min_separation})...")
    
    # 1. Load the CSV using pandas (column names are ignored to obtain the matrix)
    df = pd.read_csv(csv_path)
    probability_matrix = df.to_numpy()
    
    # Protein length (L)
    L = probability_matrix.shape[0]
    
    # 2. Create a grid of indices i, j for the entire matrix
    i_indices, j_indices = np.indices((L, L))
    
    # 3. Compute sequence separation: |i - j|
    separation = np.abs(i_indices - j_indices)
    
    # 4. Create the mask: True where separation is very small
    obvious_mask = separation < min_separation
    
    # 5. Apply the mask: set those probabilities to 0.0
    filtered_matrix = probability_matrix.copy()
    filtered_matrix[obvious_mask] = 0.0
    
    # Note: The lower triangle is also removed to avoid double-counting pairs
    # (pair 1-10 is equivalent to 10-1)
    lower_triangle_mask = np.tril(np.ones((L, L)), k=0).astype(bool)
    filtered_matrix[lower_triangle_mask] = 0.0
    
    print("Filtering complete.")
    return filtered_matrix

In [6]:
def calculate_top_l_precision(target_name, sequence):
    """
    Calculates the Top-L precision of the ESM-2 predicted contact map
    against the real contact map.
    """
    L = len(sequence)
    K = L  # We evaluate exactly Top-L
    
    if K == 0:
        print("The sequence is empty.")
        return 0.0
        
    # 1. Define paths
    real_map_path = os.path.join("real_contact_maps", f"{target_name}.csv")
    predicted_map_path = os.path.join("results_esm2", "contact_maps", f"{sequence}.csv")
    
    # Check if files exist
    if not os.path.exists(real_map_path):
        print(f"Error: Real contact map not found at {real_map_path}")
        return 0.0
    if not os.path.exists(predicted_map_path):
        print(f"Error: Predicted contact map not found at {predicted_map_path}")
        return 0.0
        
    # 2. Get the cleaned predicted matrix (calls the mask function)
    predicted_matrix = mask_obvious_contacts(predicted_map_path, min_separation=6)
    
    # 3. Load the real contact map
    df_real = pd.read_csv(real_map_path)
    real_matrix = df_real.to_numpy()
    
    # Ensure matrices are the same size
    if predicted_matrix.shape != real_matrix.shape:
        print(f"Error: Shape mismatch. Predicted {predicted_matrix.shape} vs Real {real_matrix.shape}")
        return 0.0
    
    # 4. Flatten matrices to easily extract the highest values
    flat_predicted = predicted_matrix.flatten()
    flat_real = real_matrix.flatten()
    
    # 5. Get indices of the Top K predicted probabilities
    top_k_indices = np.argsort(flat_predicted)[-K:][::-1]
    
    # 6. Count how many of those top predictions are actual contacts (value == 1)
    true_positives = np.sum(flat_real[top_k_indices] == 1)
    
    # 7. Calculate precision
    precision = true_positives / K
    
    print(f"Top-L Precision (L={L}): {precision:.4f} ({true_positives}/{K} correct contacts)")
    
    return precision

In [7]:
def calculate_spearman(target_name, sequence):

    # 1. Define paths
    real_map_path = os.path.join("real_contact_maps", f"{target_name}.csv")
    predicted_map_path = os.path.join("results_esm2", "contact_maps", f"{sequence}.csv")
    
    # 2. Check that files exist
    if not os.path.exists(real_map_path):
        print(f"Error: Real map not found at {real_map_path}")
        return 0.0
    if not os.path.exists(predicted_map_path):
        print(f"Error: Predicted map not found at {predicted_map_path}")
        return 0.0
        
    # 3. Load matrices
    df_pred = pd.read_csv(predicted_map_path)
    predicted_matrix = df_pred.to_numpy()
    
    df_real = pd.read_csv(real_map_path)
    real_matrix = df_real.to_numpy()
    
    # 4. Check dimensions
    if predicted_matrix.shape != real_matrix.shape:
        print(f"Error: Size mismatch. Predicted {predicted_matrix.shape} vs Real {real_matrix.shape}")
        return 0.0
        
    # 5. Flatten full matrices (convert NxN into a 1D vector)
    flat_predicted = predicted_matrix.flatten()
    flat_real = real_matrix.flatten()
    
    # 6. Compute Spearman correlation
    correlation, p_value = spearmanr(flat_predicted, flat_real)
    
    print(f"Spearman Correlation (Full Matrix): {correlation:.4f} (p-value: {p_value:.4e})")
    print(f"Evaluated over all {len(flat_predicted)} elements ({predicted_matrix.shape[0]}x{predicted_matrix.shape[0]}).")
    
    return correlation

Fitness calculation function

In [8]:
def fitness(target_name, sequence):

    fasta_path = create_fasta(sequence, results_fasta_dir)
    
    csv_path = generate_contact_map(fasta_path, sequence)
    
    if csv_path is None:
        return float('-inf')

    precision_l = calculate_top_l_precision(target_name, sequence)
    spearman_corr = calculate_spearman(target_name, sequence)
    
    if spearman_corr >= 0:
        fitness_value = spearman_corr + precision_l
    else:
        fitness_value = -1 * (abs(spearman_corr) + precision_l)
        
    return fitness_value

## **Genetic algorithm**

In [9]:
base_dir = os.getcwd()
genetic_functions_dir = os.path.join(base_dir, "genetic")
initial_pop_dir = os.path.join(base_dir, "initial_population")
import random

In [10]:
def tournament(population, k=2):
    contenders = random.sample(population, k)
    winner = max(contenders, key=lambda individual: individual['fitness'])
    return winner

In [11]:
POLAR_AA = ['D', 'E', 'R', 'K', 'H', 'N', 'Q', 'S', 'T', 'Y']
NONPOLAR_AA = ['A', 'G', 'V', 'L', 'I', 'P', 'F', 'M', 'W', 'C']

def mutation(sequence, mutation_prob=0.01):
    new_sequence = []
    for aa in sequence:
        if random.random() < mutation_prob:
            if aa in POLAR_AA:
                options = [option for option in POLAR_AA if option != aa]
                new_aa = random.choice(options)
            elif aa in NONPOLAR_AA:
                options = [option for option in NONPOLAR_AA if option != aa]
                new_aa = random.choice(options)
            else:
                new_aa = aa
            new_sequence.append(new_aa)
        else:
            new_sequence.append(aa)
    return "".join(new_sequence)

In [12]:
def crossover(sequence1, sequence2):
    if len(sequence1) != len(sequence2):
        raise ValueError("Sequences must have the same length.")
    length = len(sequence1)
    point1 = random.randint(0, length - 2)
    point2 = random.randint(point1 + 1, length)
    child1 = sequence1[:point1] + sequence2[point1:point2] + sequence1[point2:]
    child2 = sequence2[:point1] + sequence1[point1:point2] + sequence2[point2:]
    return child1, child2

In [13]:
import concurrent.futures

# Global dictionary to avoid evaluating the same sequence twice
fitness_cache = {}

def evaluate_sequence_esm2(sequence, target_protein):
    """
    Evaluates a sequence using the ESM-2 pipeline.
    Uses caching for efficiency.
    """
    # 1. Check if we already know the result
    if sequence in fitness_cache:
        return fitness_cache[sequence]
    
    # 2. If it's new, pass it through the integrated fitness function
    try:
        # Call the 'fitness' function created earlier
        # (This function creates the FASTA, runs ESM-2, and computes the metric)
        fit = fitness(target_protein, sequence)
        
        # Store in cache
        fitness_cache[sequence] = fit
        return fit
        
    except Exception as e:
        print(f"Error processing sequence {sequence[:10]}... : {e}")
        # IMPORTANT: Since we are maximizing, an error returns negative infinity
        return float('-inf')


In [14]:
def evaluate_batch_parallel(sequences, target_protein, max_workers=2):
    """
    Evaluates a list of sequences in parallel.
    max_workers: Number of ESM-2 processes to run on the GPU simultaneously.
    """
    results = []
    
    # ThreadPoolExecutor manages the worker queue (threads)
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        
        # Prepare all jobs by submitting them for evaluation
        futures = {
            executor.submit(evaluate_sequence_esm2, seq, target_protein): seq
            for seq in sequences
        }
        
        # As they complete (regardless of order), store the results
        for future in concurrent.futures.as_completed(futures):
            seq = futures[future]
            try:
                fit = future.result()
                results.append({'sequence': seq, 'fitness': fit})
            except Exception as exc:
                print(f"Parallel error with sequence {seq[:10]}... : {exc}")
                results.append({'sequence': seq, 'fitness': float('-inf')})
                
    return results

In [15]:
# Global dictionary to cache already computed fitness values
# to avoid evaluating the same sequence twice
fitness_cache = {}

def genetic_algorithm_esm2(target_protein, generations=300, children_count=50, p_best=200, max_gpu_workers=2, alg_folder="alg_a"):

    start_time = time.time()

    print(f"Starting ESM-2 Genetic Algorithm for: {target_protein}")
    print(f"Children per generation: {children_count}")
    
    # Set seed for reproducibility
    random.seed(26)
    
    # 1. Load initial population
    csv_path = os.path.join(initial_pop_dir, f"{target_protein}.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Initial population not found at {csv_path}")
        
    df = pd.read_csv(csv_path)
    initial_sequences = df['Sequence'].tolist()
    
    # 2. Evaluate initial population in batches
    print("Evaluating initial population in parallel (GPU)...")
    population = evaluate_batch_parallel(initial_sequences, target_protein, max_workers=max_gpu_workers)
    
    # Sort and filter unique initial population (just in case the CSV had duplicates)
    population = sorted(population, key=lambda x: x['fitness'], reverse=True)
    unique_initial = []
    seen_initial = set()
    for ind in population:
        if ind['sequence'] not in seen_initial:
            unique_initial.append(ind)
            seen_initial.add(ind['sequence'])
    population = unique_initial[:p_best]
    
    # Update initial cache
    for ind in population:
        fitness_cache[ind['sequence']] = ind['fitness']
    
    # 3. Generational loop
    for gen in range(generations):

        # Initialize historial

        fitness_vals_init = [ind['fitness'] for ind in population]
        history_stats = [{
            'generation': 0,
            'best_fitness': np.max(fitness_vals_init),
            'mean_fitness': np.mean(fitness_vals_init),
            'median_fitness': np.median(fitness_vals_init)
        }]


        
        # Selection pressure transition
        current_k = 2 if gen < 20 else 3
        print(f"\n--- Generation {gen + 1}/{generations} | Tournament k={current_k} ---")
        
        # STEP A: Select ALL parents first via tournament
        parents = []
        for _ in range(children_count):
            parents.append(tournament(population, k=current_k))
            
        # STEP B: Perform crossover and mutation
        children_sequences_raw = []
        for i in range(0, children_count, 2):
            parent1 = parents[i]
            # Handle odd children_count smoothly
            parent2 = parents[i+1] if i+1 < len(parents) else parents[0] 
            
            # Crossover
            child_seq1, child_seq2 = crossover(parent1['sequence'], parent2['sequence'])
            
            # Mutation
            mutated_child1 = mutation(child_seq1)
            mutated_child2 = mutation(child_seq2)
            
            children_sequences_raw.extend([mutated_child1, mutated_child2])
            
        # STEP C: Filter sequences using cache
        unique_to_evaluate = [seq for seq in children_sequences_raw if seq not in fitness_cache]
        
        # Retrieve already known ones
        children_population = [
            {'sequence': seq, 'fitness': fitness_cache[seq]}
            for seq in children_sequences_raw if seq in fitness_cache
        ]
        
        # Evaluate NEW sequences in parallel on GPU
        if unique_to_evaluate:
            print(f"Sending batch of {len(set(unique_to_evaluate))} new unique sequences to GPU...")
            new_evaluations = evaluate_batch_parallel(
                list(set(unique_to_evaluate)), target_protein, max_workers=max_gpu_workers
            )
            children_population.extend(new_evaluations)
            
            # Update cache with new results
            for ind in new_evaluations:
                fitness_cache[ind['sequence']] = ind['fitness']
                
        # STEP D: (μ + λ) scheme - Merge parents and children and select the best
        combined_population = population + children_population
        combined_population = sorted(combined_population, key=lambda x: x['fitness'], reverse=True)
        
        
        unique_population = []
        seen_sequences = set()
        
        for ind in combined_population:
            if ind['sequence'] not in seen_sequences:
                unique_population.append(ind)
                seen_sequences.add(ind['sequence'])
                
            
            if len(unique_population) == p_best:
                break
        
        # Only the best individuals
        population = unique_population


        # Only the best individuals
        population = unique_population
        
        current_fitness_vals = [ind['fitness'] for ind in population]
        best_fit = np.max(current_fitness_vals)
        mean_fit = np.mean(current_fitness_vals)
        median_fit = np.median(current_fitness_vals)
        
        history_stats.append({
            'generation': gen + 1,
            'best_fitness': best_fit,
            'mean_fitness': mean_fit,
            'median_fitness': median_fit
        })
        
        print(f"Best: {best_fit:.4f} | Mean: {mean_fit:.4f} | Median: {median_fit:.4f} | Unique: {len(population)}")


        
        print(f"Best current fitness: {population[0]['fitness']:.4f} | Total unique individuals: {len(population)}")
        
    print("\nEvolution finished")
    
    # 4. Save final results
    df_results = pd.DataFrame(population)
    
    # Definir la ruta de la subcarpeta y crearla si no existe
    target_dir = os.path.join(esm2_dir, alg_folder)
    os.makedirs(target_dir, exist_ok=True)
    
    file_name = f"genetic_results_esm2_{target_protein}.csv"
    save_path = os.path.join(target_dir, file_name)
    
    df_results.to_csv(save_path, index=False)
    print(f"Final population successfully saved at:\n{save_path}")

    # Total time
    end_time = time.time()
    total_time_seconds = end_time - start_time
    
    time_file_name = f"time_{target_protein}.txt"
    time_save_path = os.path.join(target_dir, time_file_name)
    
    with open(time_save_path, "w") as file:
        file.write(f"Sequence: {target_protein}\n")
        file.write(f"Total time: {total_time_seconds:.2f} seconds \n")
        
    print(f"Execution time saved at:\n{time_save_path}")


    df_stats = pd.DataFrame(history_stats)
    stats_file_name = f"statistics_{target_protein}.csv"
    stats_save_path = os.path.join(target_dir, stats_file_name)
    
    df_stats.to_csv(stats_save_path, index=False)
    print(f"Generation statistics successfully saved at:\n{stats_save_path}")
    
    return population

## **Execution**

In [16]:

target_protein_name = "1aay" 
'''
final_population = genetic_algorithm_esm2(
    target_protein = target_protein_name,
    generations=30,       # Total number of generations
    children_count=30,     # Number of children generated per generation
    p_best=25,            # Population size that survives
    max_gpu_workers=4,      # Sequences evaluated simultaneously on the GPU
    alg_folder="alg_a"
)
'''

'\nfinal_population = genetic_algorithm_esm2(\n    target_protein = target_protein_name,\n    generations=30,       # Total number of generations\n    children_count=30,     # Number of children generated per generation\n    p_best=25,            # Population size that survives\n    max_gpu_workers=4,      # Sequences evaluated simultaneously on the GPU\n    alg_folder="alg_a"\n)\n'

In [53]:
def plot_fitness_evolution(target_protein, alg_folder="alg_a"):
    import os
    import pandas as pd
    import matplotlib.pyplot as plt

    # 1. Define paths
    esm2_dir = "results_esm2"
    target_dir = os.path.join(esm2_dir, alg_folder)
    stats_file = f"statistics_{target_protein}.csv"
    stats_path = os.path.join(target_dir, stats_file)

    # 2. Check file exists
    if not os.path.exists(stats_path):
        print(f"Error: File not found at {stats_path}")
        return

    # 3. Load data
    df = pd.read_csv(stats_path)

    # 4. Create plot
    plt.figure(figsize=(10, 6))

    plt.plot(df['generation'], df['best_fitness'],
             marker='8', linewidth=2, markersize=3,
             color='#237227',
             label='Best Fitness')

    plt.plot(df['generation'], df['median_fitness'],
             marker='8', linewidth=2, markersize=4,
             color='#261CC1',
             label='Median Fitness')

    # 5. Format plot
    plt.title(
        f'Best vs Median fitness - {target_protein.upper()}',
        fontsize=14,
        fontweight='bold'
    )
    plt.xlabel('Generation', fontsize=12)
    plt.ylabel('Fitness', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=12)

    # 6. Save plot
    plot_file = f"plot_evolution_best_median_{target_protein}.png"
    plot_path = os.path.join(target_dir, plot_file)

    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"Plot saved at:\n{plot_path}")

    plt.close()

In [63]:
plot_fitness_evolution("1p9n", alg_folder="alg_a")
#median_fitness

Plot saved at:
results_esm2/alg_a/plot_evolution_best_median_1p9n.png
